# Kalman localization

Estimate position and velocity from noisy world-frame measurements.

Run with the repository virtual environment. These are synthetic research experiments; units and assumptions are explicit below.

## Measurement process

The filter assumes constant velocity with acceleration process noise, independent of bicycle dynamics.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from autonomous_vehicles.simulation.localization import (
    ConstantVelocityFilter,
    LocalizationConfig,
    noisy_positions,
)

dt = 0.1
times = np.arange(100) * dt
truth = np.column_stack((times * 2, times * 0.5))
measurements = noisy_positions(truth, standard_deviation=0.4, seed=42)
config = LocalizationConfig(position_noise=0.4)

## Filter

Prediction propagates covariance; measurement updates use a Joseph covariance formula.

In [ ]:
estimator = ConstantVelocityFilter(config=config)
estimates, uncertainty, accepted = [], [], []
for measurement in measurements:
    estimator.predict(dt)
    estimate = estimator.update(measurement)
    estimates.append(estimate.state)
    uncertainty.append(np.sqrt(np.diag(estimate.covariance)))
    accepted.append(estimate.accepted)
estimates = np.array(estimates)
uncertainty = np.array(uncertainty)
print("Accepted:", sum(accepted), "/", len(accepted))

## Position estimate

Plot measurements separately from the filtered trajectory.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(measurements[:, 0], measurements[:, 1], s=8, alpha=0.4, label="Noisy positions")
ax.plot(truth[:, 0], truth[:, 1], label="Truth")
ax.plot(estimates[:, 0], estimates[:, 1], label="Filtered")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.legend()
plt.show()

## Uncertainty and velocity

Covariance is model-based uncertainty and requires calibration against repeated trials.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].plot(times, estimates[:, 0] - truth[:, 0], label="x error")
axes[0].fill_between(
    times, -2 * uncertainty[:, 0], 2 * uncertainty[:, 0], alpha=0.2, label="±2 std"
)
axes[0].set_ylabel("Position error (m)")
axes[1].plot(times, estimates[:, 2], label="vx")
axes[1].plot(times, estimates[:, 3], label="vy")
for ax in axes:
    ax.set_xlabel("Time (s)")
    ax.legend()
plt.show()

## Gate an outlier

A rejected measurement leaves the predicted state unchanged.

In [ ]:
before = estimator.snapshot()
outlier = estimator.update([1000, 1000])
assert not outlier.accepted
np.testing.assert_array_equal(before.state, outlier.state)
print("Outlier squared innovation distance:", outlier.innovation_squared_distance)
raw_rmse = np.sqrt(np.mean((measurements[20:] - truth[20:]) ** 2))
filtered_rmse = np.sqrt(np.mean((estimates[20:, :2] - truth[20:]) ** 2))
print({"raw_rmse_m": raw_rmse, "filtered_rmse_m": filtered_rmse})

## Research record

Record the seed, configuration, dependency lock and observed failures with each experiment. Results here demonstrate the implemented model under its stated assumptions. They do not establish performance outside this synthetic distribution.